In [3]:
# 2. 필요한 모듈 임포트
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
from plotly.subplots import make_subplots

# 3. 데이터 준비
from services.data_preparer import prepare_basic_proposal_data

# --- 테스트 실행 ---

# 1. app.py의 역할 (1): 데이터 준비
print("Step 1: `data_preparer`를 통해 분석용 데이터를 준비합니다...")
data_bundle_full = prepare_basic_proposal_data() 
data_bundle = data_bundle_full.get("data_bundle", {})
order_map = data_bundle_full.get("order_map", {})
print(" -> 데이터 준비 완료!")


# 2. app.py의 역할 (2): 사용자 선택 시뮬레이션
DIMENSION_CONFIG = {
    '부서별': {'type': 'hierarchical', 'top': 'DIVISION_NAME', 'sub': 'OFFICE_NAME'},
    '직무별': {'type': 'hierarchical', 'top': 'JOB_L1_NAME', 'sub': 'JOB_L2_NAME'},
    '직위직급별': {'type': 'flat', 'col': 'POSITION_NAME'},
    '성별': {'type': 'flat', 'col': 'GENDER'},
    '연령별': {'type': 'flat', 'col': 'AGE_BIN'},
    '경력연차별': {'type': 'flat', 'col': 'CAREER_BIN'},
    '연봉구간별': {'type': 'flat', 'col': 'SALARY_BIN'},
    '지역별': {'type': 'flat', 'col': 'REGION_CATEGORY'},
    '계약별': {'type': 'flat', 'col': 'CONT_CATEGORY'}
}

selected_dimension_ui = '부서별'
drilldown_selection = '전체' # 'Planning Division' 등으로 변경 가능

print(f"Step 2: 사용자가 '{selected_dimension_ui}' 차원을, '{drilldown_selection}' 그룹으로 선택했습니다.")


# 3. ipynb 테스트용 헬퍼 함수 정의
# (basic_proposal_view.py의 _build_tab_content 헬퍼 함수 로직을 그대로 가져온 것)
def _generate_fig_and_df(period_agg_name, period_source_col, tail_n):
    """
    ipynb 테스트를 위해, 탭 1개의 로직을 시뮬레이션하고 fig, df를 반환합니다.
    """
    config = DIMENSION_CONFIG.get(selected_dimension_ui, {})
    dimension_col = config.get('top', config.get('col'))
    
    dimension_data = data_bundle.get(selected_dimension_ui, data_bundle.get('전체', {}))
    summary_df_for_agg = dimension_data.get(period_agg_name, pd.DataFrame())
    overall_summary_df_for_agg = data_bundle.get('전체', {}).get(period_agg_name, pd.DataFrame())

    if summary_df_for_agg.empty or overall_summary_df_for_agg.empty:
        return go.Figure().update_layout(title_text="데이터 없음"), pd.DataFrame()

    # 기간(PERIOD) 컬럼 생성
    for df in [summary_df_for_agg, overall_summary_df_for_agg]:
        if period_agg_name == 'quarterly':
            df['PERIOD'] = df[period_source_col].apply(lambda q: f"{q.year}년 {q.quarter}분기")
        elif period_agg_name == 'yearly':
            df['PERIOD'] = df[period_source_col].apply(lambda y: f"{y}년")
        else: # monthly
            df['PERIOD'] = df[period_source_col].dt.strftime('%Y년 %m월')
    
    # 그래프용 데이터(plot_df) 최종 선택
    if drilldown_selection == '전체' or not dimension_col:
        plot_df = overall_summary_df_for_agg.tail(tail_n)
        title = f"[{period_agg_name.upper()}] 인원 변동 현황"
    else:
        plot_df = summary_df_for_agg[summary_df_for_agg[dimension_col] == drilldown_selection].tail(tail_n)
        title = f"[{drilldown_selection} - {period_agg_name.upper()}] 인원 변동 현황"

    # 그래프 생성
    if plot_df.empty:
        fig = go.Figure().update_layout(title_text=f"'{drilldown_selection}'에 대한 데이터가 없습니다.")
    else:
        fig = make_subplots(specs=[[{"secondary_y": True}]])
        fig.add_trace(go.Bar(x=plot_df['PERIOD'], y=plot_df['NEW_HIRES'], name='입사자', marker_color='blue'), secondary_y=False)
        fig.add_trace(go.Bar(x=plot_df['PERIOD'], y=plot_df['LEAVERS'], name='퇴사자', marker_color='red'), secondary_y=False)
        fig.add_trace(go.Scatter(x=plot_df['PERIOD'], y=plot_df['HEADCOUNT'], name='총원', mode='lines+markers+text', text=plot_df['HEADCOUNT'], textposition='top center', line=dict(color='black')), secondary_y=True)
        max_val = max(plot_df['NEW_HIRES'].max(), plot_df['LEAVERS'].max())
        y1_range = [0, max_val * 1.5 if max_val > 0 else 10]
        fig.update_layout(template='plotly', title_text=title, xaxis_title='기간', font_size=14, height=500, barmode='group')
        fig.update_yaxes(title_text="입사/퇴사자 수", secondary_y=False, range=y1_range)
        fig.update_yaxes(title_text="총원", secondary_y=True, rangemode='tozero')
    
    # 요약 테이블(aggregate_df) 생성
    aggregate_df = pd.DataFrame()
    if not summary_df_for_agg.empty and not overall_summary_df_for_agg.empty and dimension_col:
        agg_by_dim = summary_df_for_agg.pivot_table(index='PERIOD', columns=dimension_col, values='HEADCOUNT', aggfunc='last')
        agg_overall = overall_summary_df_for_agg.pivot_table(index='PERIOD', values='HEADCOUNT', aggfunc='last').rename(columns={'HEADCOUNT': '전체'})
        aggregate_df = pd.concat([agg_overall, agg_by_dim], axis=1).fillna(0).astype(int)
        
        ordered_categories = order_map.get(dimension_col, sorted(summary_df_for_agg[dimension_col].unique()))
        cols_ordered = ['전체'] + [col for col in ordered_categories if col in aggregate_df.columns]
        
        final_cols = [col for col in cols_ordered if col in aggregate_df.columns]
        aggregate_df = aggregate_df[final_cols]
        aggregate_df = aggregate_df.tail(tail_n)

    return fig, aggregate_df

# --- 결과 확인 ---
# 4. 각 탭의 로직을 개별적으로 실행하고 결과 확인

print("\n--- [결과 1] 월별(Monthly) 탭 시뮬레이션 ---")
fig_m, df_m = _generate_fig_and_df(period_agg_name='monthly', period_source_col='PERIOD_DT', tail_n=12)
fig_m.show()
display(df_m)

print("\n--- [결과 2] 분기별(Quarterly) 탭 시뮬레이션 ---")
fig_q, df_q = _generate_fig_and_df(period_agg_name='quarterly', period_source_col='QUARTER', tail_n=12)
fig_q.show()
display(df_q)

print("\n--- [결과 3] 연간(Yearly) 탭 시뮬레이션 ---")
fig_y, df_y = _generate_fig_and_df(period_agg_name='yearly', period_source_col='YEAR', tail_n=10)
fig_y.show()
display(df_y)

Step 1: `data_preparer`를 통해 분석용 데이터를 준비합니다...
 -> 데이터 준비 완료!
Step 2: 사용자가 '부서별' 차원을, '전체' 그룹으로 선택했습니다.

--- [결과 1] 월별(Monthly) 탭 시뮬레이션 ---


,전체,Planning Division,Sales Division,Development Division,Operating Division
PERIOD,,,,,
2024년 11월,518,124,143,114,137
2024년 12월,513,122,141,112,138
2025년 01월,514,121,145,111,137
2025년 02월,508,122,141,110,135
2025년 03월,510,122,140,114,134
2025년 04월,508,122,141,114,131
2025년 05월,505,120,140,115,130
2025년 06월,505,119,141,115,130
2025년 07월,502,122,140,115,125



--- [결과 2] 분기별(Quarterly) 탭 시뮬레이션 ---


,전체,Planning Division,Sales Division,Development Division,Operating Division
PERIOD,,,,,
2023년 1분기,490,112,141,108,129
2023년 2분기,496,114,144,109,129
2023년 3분기,486,110,143,105,128
2023년 4분기,498,115,144,107,132
2024년 1분기,506,121,143,107,135
2024년 2분기,504,121,143,110,130
2024년 3분기,516,124,146,113,133
2024년 4분기,513,122,141,112,138
2025년 1분기,510,122,140,114,134



--- [결과 3] 연간(Yearly) 탭 시뮬레이션 ---


,전체,Planning Division,Sales Division,Development Division,Operating Division
PERIOD,,,,,
2016년,326,69,88,77,92
2017년,364,89,99,83,93
2018년,396,98,115,87,96
2019년,428,104,123,86,115
2020년,453,114,118,99,122
2021년,471,118,128,108,117
2022년,486,115,139,105,127
2023년,498,115,144,107,132
2024년,513,122,141,112,138
